# GOL-08b : le certificat d'impossibilite du translateur, prouve en Lean

**Navigation** : [<< GOL-08 Loi II sur substrat Life](GOL-08-Loi-II-Translateur-Life.ipynb) | [Index](../README.md) | [GOL-09 Perplexite structurelle >>](GOL-09-Perplexite-Structurelle-Hashlife.ipynb)

**Kernel** : **Lean 4 (WSL) natif** — chaque cellule de code est du Lean 4 exécuté directement. Le module compagnon est [`Conway.Life.Translateur`](../../../SymbolicAI/Lean/conway_lean/Conway/Life/Translateur.lean) du lake `conway_lean` (jumeau EN : `Translateur_en.lean`).

> GOL-08 a construit, en Python, le synthétiseur de translateurs et son certificat d'impossibilité **scopé à une famille finie**. Sa conclusion annonçait : « la certification **formelle** (jambe Lean) est explicitement hors scope — si elle vient, ce sera un variant `-b` ». Ce carnet est ce variant : la même démarche — famille finie, témoin, impossibilité, contraste de frontière — rejouée avec le **kernel Lean comme moteur de vérité**, chaque verdict étant un théorème du lake.

## 1. Ce que le `-b` change

En Python, le certificat de GOL-08 est un **balayage exhaustif** : le synthétiseur essaie chaque pattern de la famille et conclut « aucun ne translate ». C'est une preuve aussi solide que le code qui l'exécute — mais le code lui-même n'est vérifié par personne.

Ici, le balayage devient une **expression booléenne que le kernel Lean réduit**. `noTranslator b k v p` parcourt la famille et rend `false` dès qu'un pattern contredit le verdict ; quand le kernel le réduit à `true` — et a fortiori quand `theorem ... := by decide` compile — l'échec exhaustif est **prouvé**, pas constaté. La différence est le statut : un test peut avoir un bug d'énumération, un théorème ne peut pas avoir eu raison par accident.

Le prix de ce statut est la **taille** : le kernel décide des familles de quelques centaines de patterns, pas de dizaines de milliers. La section 5 chiffre cette frontière honnêtement.

## 2. Le module `Conway.Life.Translateur`

L'API du module est volontairement le miroir du carnet Python :

| Concept GOL-08 (Python) | Ici (Lean) |
|---|---|
| boîte `b x b` | `boxCells b : List (Int × Int)` |
| famille des patterns de 1 à `k` cellules | `family b k : List Grid` |
| synthétiseur trouve un translateur | `hasTranslator b k v p : Bool` |
| certificat « aucun translateur » | `noTranslator b k v p : Bool` |
| vecteur `(dx, dy)`, période `p` | `v : Int × Int`, `p : Nat` |

In [1]:
import Conway.Life.Translateur
import Conway.Life.Spaceships

open Conway.Life

-- Les quatre pieces du module : signatures.
#check @boxCells
#check @family
#check @noTranslator
#check @hasTranslator

import Conway.Life.Translateur
import Conway.Life.Spaceships

open Conway.Life

-- Les quatre pieces du module : signatures.
#check @boxCells
──────▶  boxCells : ℕ → List (ℤ × ℤ)
#check @family
──────▶  family : ℕ → ℕ → List Grid
#check @noTranslator
──────▶  noTranslator : ℕ → ℕ → ℤ × ℤ → ℕ → Bool
#check @hasTranslator
──────▶  hasTranslator : ℕ → ℕ → ℤ × ℤ → ℕ → Bool
--% env 0

Raw input:
{"cmd": "import Conway.Life.Translateur\nimport Conway.Life.Spaceships\n\nopen Conway.Life\n\n-- Les quatre pieces du module : signatures.\n#check @boxCells\n#check @family\n#check @noTranslator\n#check @hasTranslator"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "boxCells : ℕ → List (ℤ × ℤ)"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data": "family : ℕ → ℕ → List Grid"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data": "noTranslator : ℕ → ℕ → ℤ × ℤ → ℕ → Bool"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data": "hasTranslator : ℕ → ℕ → ℤ × ℤ → ℕ → Bool"}],
 "env": 0}

### 2.1 La boîte et sa famille

`boxCells b` énumère les `b * b` cellules de la boîte en ordre lexicographique — l'ordre canonique des grilles de `Conway.Life` :

In [2]:
-- Les positions de la boite 3x3, dans l'ordre canonique.
#eval boxCells 3

-- Les positions de la boite 3x3, dans l'ordre canonique.
#eval boxCells 3
─────▶  [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
--% env 1

Raw input:
{"cmd": "-- Les positions de la boite 3x3, dans l'ordre canonique.\n#eval boxCells 3", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]"}],
 "env": 1}

In [3]:
-- Tailles des familles utilisees dans ce carnet :
--   (3, 5) : patterns de taille 1 a 5 dans la boite 3x3
--   (3, 3) : patterns de taille 1 a 3 dans la boite 3x3
#eval (family 3 5).length
#eval (family 3 3).length

-- Tailles des familles utilisees dans ce carnet :
--   (3, 5) : patterns de taille 1 a 5 dans la boite 3x3
--   (3, 3) : patterns de taille 1 a 3 dans la boite 3x3
#eval (family 3 5).length
─────▶  381
#eval (family 3 3).length
─────▶  129
--% env 2

Raw input:
{"cmd": "-- Tailles des familles utilisees dans ce carnet :\n--   (3, 5) : patterns de taille 1 a 5 dans la boite 3x3\n--   (3, 3) : patterns de taille 1 a 3 dans la boite 3x3\n#eval (family 3 5).length\n#eval (family 3 3).length", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "381"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "129"}],
 "env": 2}

**Lecture du résultat.** La famille (3, 5) compte exactement `381` patterns — la somme des coefficients binomiaux `C(9,1) + C(9,2) + C(9,3) + C(9,4) + C(9,5) = 9 + 36 + 84 + 126 + 126` — et la famille (3, 3) en compte `129 = 9 + 36 + 84`. Chaque élément est une sous-liste de `boxCells b` : comme la boîte est triée et sans doublon, **chaque pattern de la famille est déjà canonique** sans repasser par `sortDedup`. C'est ce qui rend l'énumération à la fois complète (toute sous-liste non vide de la bonne taille y est) et sans doublons (deux sous-listes distinctes de la boîte sont deux grilles distinctes).

## 3. Le témoin : le glider retrouvé par énumération

Le premier geste de GOL-08 est le même qu'ici : vérifier que le synthétiseur **trouve** quelque chose sur un cas connu, avant de lui faire certifier des échecs. Le cas connu, c'est le glider — taille 5, translateur `(1, -1)` en 4 générations, déjà prouvé dans `Conway.Life` (`glider_spaceship`).

In [4]:
-- La famille (3, 5) contient-elle un translateur (1, -1)/4 ?
#eval hasTranslator 3 5 (1, -1) 4

-- La famille (3, 5) contient-elle un translateur (1, -1)/4 ?
#eval hasTranslator 3 5 (1, -1) 4
─────▶  true
--% env 3

Raw input:
{"cmd": "-- La famille (3, 5) contient-elle un translateur (1, -1)/4 ?\n#eval hasTranslator 3 5 (1, -1) 4", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "true"}],
 "env": 3}

In [5]:
-- Le lake l'a prouve : le temoin est legitime (glider dans la famille)
-- et le certificat de presence tient.
#check @translator_witness_glider
#check @glider_mem_family

-- Le lake l'a prouve : le temoin est legitime (glider dans la famille)
-- et le certificat de presence tient.
#check @translator_witness_glider
──────▶  translator_witness_glider : hasTranslator 3 5 (1, -1) 4 = true
#check @glider_mem_family
──────▶  glider_mem_family : glider ∈ family 3 5
--% env 4

Raw input:
{"cmd": "-- Le lake l'a prouve : le temoin est legitime (glider dans la famille)\n-- et le certificat de presence tient.\n#check @translator_witness_glider\n#check @glider_mem_family", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "translator_witness_glider : hasTranslator 3 5 (1, -1) 4 = true"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "glider_mem_family : glider ∈ family 3 5"}],
 "env": 4}

**Lecture du résultat.** `hasTranslator 3 5 (1, -1) 4` rend `true` : en parcourant les 381 patterns, le kernel en trouve un — le glider — dont `evolve 4` égale `shift (1, -1)`. L'évaluation s'arrête au premier témoin, exactement comme le synthétiseur Python s'arrêtait à la première configuration valide. Les deux `#check` montrent que le lake va plus loin que le constat d'exécution : `translator_witness_glider` est un **théorème** (`hasTranslator 3 5 (1, -1) 4 = true`, prouvé par `decide`), et `glider_mem_family` garantit séparément que le glider est bien un élément de la famille énumérée — le témoin n'est pas un artefact d'énumération.

## 4. La Loi II et le certificat d'impossibilité

La Loi II du Jeu de Vie borne la vitesse des translateurs : l'« information » vit au plus à `c` — une cellule par génération — et un déplacement **orthogonal** est borné par `c/2`. Le cas-test de GOL-08 est `(dx, dy) = (2, 0)` avec `p = 3` : vitesse `2/3 c`, au-delà de la limite orthogonale. Aucun pattern — aucune taille, aucune boîte — ne peut s'y soustraire ; le certificat scopé le vérifie sur la famille (3, 5) :

In [6]:
-- Certificat d'impossibilite : AUCUN pattern de taille 1 a 5 dans la
-- boite 3x3 ne se translate de (2, 0) en 3 generations.
#eval noTranslator 3 5 (2, 0) 3

-- Certificat d'impossibilite : AUCUN pattern de taille 1 a 5 dans la
-- boite 3x3 ne se translate de (2, 0) en 3 generations.
#eval noTranslator 3 5 (2, 0) 3
─────▶  true
--% env 5

Raw input:
{"cmd": "-- Certificat d'impossibilite : AUCUN pattern de taille 1 a 5 dans la\n-- boite 3x3 ne se translate de (2, 0) en 3 generations.\n#eval noTranslator 3 5 (2, 0) 3", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "true"}],
 "env": 5}

**Lecture du résultat.** `true` : les 381 patterns — glider compris, qui est dans la famille — ont été évalués un à un, et aucun ne reparaît translaté de `(2, 0)` après 3 générations. Dans le lake, ce n'est pas une évaluation mais un théorème : `no_translator_two_thirds_c : noTranslator 3 5 (2, 0) 3 = true`, prouvé par `decide` — le kernel a réduit l'énoncé complet à `true` **à la compilation**. La vitesse `2/3 c` viole la borne orthogonale `c/2` ; ici la violation n'est pas un argument physique cité, c'est un échec constaté pattern par pattern sur toute une famille.

## 5. La frontière honnête : Python décide plus grand, Lean décide plus sûr

| | GOL-08 (Python) | GOL-08b (Lean, ce carnet) |
|---|---|---|
| boîte | 5 x 5 | 3 x 3 |
| famille (b, k=5) | 68 405 candidats | 381 candidats |
| famille (b, k=3) | 2 625 candidats | 129 candidats |
| statut du verdict | balayage exécuté (code non vérifié) | théorème `by decide` |
| impossibilité (2, 0)/3 | certifiée sur la boîte 5x5 | certifiée sur la boîte 3x3 |

L'énoncé Lean est **strictement plus faible** que son homologue Python : couvrir la boîte 3x3 ne dit rien des patterns à cheval sur une boîte 5x5. Le budget de réduction du kernel — chaque `decide` évalue toutes les générations intermédiaires de chaque candidat — borne aujourd'hui la famille à quelques centaines de patterns. Le module énonce cette frontière dans sa propre documentation : aucun théorème ci-dessus ne prétend couvrir la boîte 5x5. Les deux carnet sont complémentaires : Python montre que la famille peut être grande, Lean montre que le verdict peut être un théorème.

## 6. La frontière de famille : ce que la taille interdit, la règle ne l'interdit pas

Le contraste final de GOL-08 : une famille peut être vide de translateurs alors que le vecteur cherché est réalisable **hors de la famille**. Version formelle — la famille (3, 3) (patterns de taille au plus 3) contre le vaisseau LWSS du lake :

In [7]:
-- Aucun pattern de taille au plus 3 dans la boite 3x3 ne translate (0, 2)/4...
#eval noTranslator 3 3 (0, 2) 4

-- ...pourtant le LWSS (taille 9, hors famille par la seule taille)
-- realise exactement ce deplacement : le lake l'a prouve.
#check @lwss_spaceship

-- Aucun pattern de taille au plus 3 dans la boite 3x3 ne translate (0, 2)/4...
#eval noTranslator 3 3 (0, 2) 4
─────▶  true

-- ...pourtant le LWSS (taille 9, hors famille par la seule taille)
-- realise exactement ce deplacement : le lake l'a prouve.
#check @lwss_spaceship
──────▶  lwss_spaceship : isSpaceship lwss 4 (0, 2) = true
--% env 6

Raw input:
{"cmd": "-- Aucun pattern de taille au plus 3 dans la boite 3x3 ne translate (0, 2)/4...\n#eval noTranslator 3 3 (0, 2) 4\n\n-- ...pourtant le LWSS (taille 9, hors famille par la seule taille)\n-- realise exactement ce deplacement : le lake l'a prouve.\n#check @lwss_spaceship", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "lwss_spaceship : isSpaceship lwss 4 (0, 2) = true"}],
 "env": 6}

**Lecture du résultat.** Le `#eval` rend `true` (aucun des 129 patterns ne fait `(0, 2)` en 4 générations — théorème `no_small_orthogonal_translator` dans le lake) pendant que `lwss_spaceship : isSpaceship lwss 4 (0, 2) = true` atteste le contraire… pour un pattern de taille 9. Les deux énoncés coexistent parce qu'ils ne parlent pas du même monde : le certificat dit « **dans cette famille**, aucun », et la famille s'arrête aux patterns de taille 3. C'est la leçon méthodologique de GOL-08, désormais avec preuves des deux côtés de la frontière : un certificat d'impossibilité vaut exactement sa portée, jamais un de plus.

## 7. Le lemme de portée : le certificat dit précisément ce qu'il couvre

Ce qui empêche le certificat d'être surinterprété, c'est qu'il est accompagné d'un lemme qui délimite la famille **dans l'énoncé formel lui-même** :

In [8]:
-- Tout pattern couvert vit dans la boite b x b : le certificat est
-- precise, ni plus large, ni plus etroit.
#check @family_scope

-- Tout pattern couvert vit dans la boite b x b : le certificat est
-- precise, ni plus large, ni plus etroit.
#check @family_scope
──────▶  @family_scope : ∀ {b k : ℕ} {g : Grid}, g ∈ family b k → ∀ {p : ℤ × ℤ}, p ∈ g → 0 ≤ p.1 ∧ p.1 < ↑b ∧ 0 ≤ p.2 ∧ p.2 < ↑b
--% env 7

Raw input:
{"cmd": "-- Tout pattern couvert vit dans la boite b x b : le certificat est\n-- precise, ni plus large, ni plus etroit.\n#check @family_scope", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@family_scope : ∀ {b k : ℕ} {g : Grid}, g ∈ family b k → ∀ {p : ℤ × ℤ}, p ∈ g → 0 ≤ p.1 ∧ p.1 < ↑b ∧ 0 ≤ p.2 ∧ p.2 < ↑b"}],
 "env": 7}

**Lecture du résultat.** `family_scope` dit : si `g` est dans la famille `(b, k)` et `p` est une cellule de `g`, alors les coordonnées de `p` sont dans `[0, b)`. Combiné à `noTranslator`, le théorème complet se lit : « aucun pattern **dont toutes les cellules vivent dans la boîte `b x b`** et **comportant entre 1 et `k` cellules** n'est un translateur `v`/`p` ». Chaque borne de l'énoncé — la boîte, la taille — est portée par un énoncé formel ; il ne reste aucune place pour étendre le verdict par suggestion.

## 8. Conventions de coordonnées : lire les vecteurs de GOL-08 ici

Le carnet Python compte `dy` positif **vers le bas** (convention écran) ; le lake `conway_lean` compte `dy` positif **vers le haut** (convention mathématique). Les correspondances des cas du carnet :

| Cas GOL-08 (Python) | Ici (Lean) | Verdict |
|---|---|---|
| `(3, 5, 1, 1, 4)` — témoin | `hasTranslator 3 5 (1, -1) 4` | `true` (glider) |
| `(5, 5, 2, 0, 3)` — impossibilité 2/3 c | `noTranslator 3 5 (2, 0) 3` | `true` (boîte 3x3) |
| `(5, 5, 2, 0, 4)` — contraste famille | `noTranslator 3 3 (0, 2) 4` | `true` (LWSS hors famille) |

Un vecteur Python `(dx, dy)` se lit ici `(dx, -dy)` ; les vecteurs purement horizontaux `(dx, 0)` ou `(0, dy)` ne changent que par le signe, jamais par la norme.

## 9. Exercices

Trois exercices pour manipuler le module. Chaque cellule **compile et s'exécute** telle quelle ; à vous de remplacer la partie signalée `-- TODO`.

### Exercice 1 — Détecter une vitesse superluminique

La Loi II dit qu'un translateur `(dx, dy)` de période `p` ne peut pas dépasser `c` : en prenant la composante de plus grand module, `2 * max(|dx|, |dy|) > p` signale un vecteur physiquement impossible — le facteur 2 évite la division rationnelle.

**A vous** : écrivez `overLightSpeed (v : Int × Int) (p : Nat) : Bool` qui rend `true` ssi ce seuil est dépassé.

In [9]:
-- Exercice 1 : true ssi 2 * max(|v.1|, |v.2|) > p
-- TODO etudiant : remplacer le corps ci-dessous
-- Indice : Int.natAbs donne la valeur absolue en Nat ; max : Nat -> Nat -> Nat ;
-- Etape 1 : max (Int.natAbs v.1) (Int.natAbs v.2)
-- Etape 2 : comparer 2 * ce maximum a p (decide ou < en Bool).
def overLightSpeed (v : Int × Int) (p : Nat) : Bool :=
  false

-- Attendus : (2, 0)/3 depasse c ; le glider (1, -1)/4 non ; le LWSS (0, 2)/4 non.
#eval overLightSpeed (2, 0) 3
#eval overLightSpeed (1, -1) 4
#eval overLightSpeed (0, 2) 4

-- Exercice 1 : true ssi 2 * max(|v.1|, |v.2|) > p
-- TODO etudiant : remplacer le corps ci-dessous
-- Indice : Int.natAbs donne la valeur absolue en Nat ; max : Nat -> Nat -> Nat ;
-- Etape 1 : max (Int.natAbs v.1) (Int.natAbs v.2)
-- Etape 2 : comparer 2 * ce maximum a p (decide ou < en Bool).
def overLightSpeed (v : Int × Int) (p : Nat) : Bool :=
                    ─▶ 🟨 Variable name `v` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _v

Note: This linter can be disabled with `set_option linter.unusedVariables false`
                                    ─▶ 🟨 Variable name `p` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _p

Note: This linter can be disabled with `set_option linter.unusedVariables false`
  false

-- Attendus : (2, 0)/3 depasse c ; le glider (1, -1)/4 non ; le LWSS (0, 2)/4 non.
#eval overLightSpeed (2, 0) 3
─────▶  false
#eval overLightSpeed (1, -1) 4
─────▶  false
#eval overLightSpeed (0, 2) 4
─────▶  false
--% env 8

Raw input:
{"cmd": "-- Exercice 1 : true ssi 2 * max(|v.1|, |v.2|) > p\n-- TODO etudiant : remplacer le corps ci-dessous\n-- Indice : Int.natAbs donne la valeur absolue en Nat ; max : Nat -> Nat -> Nat ;\n-- Etape 1 : max (Int.natAbs v.1) (Int.natAbs v.2)\n-- Etape 2 : comparer 2 * ce maximum a p (decide ou < en Bool).\ndef overLightSpeed (v : Int \u00d7 Int) (p : Nat) : Bool :=\n  false\n\n-- Attendus : (2, 0)/3 depasse c ; le glider (1, -1)/4 non ; le LWSS (0, 2)/4 non.\n#eval overLightSpeed (2, 0) 3\n#eval overLightSpeed (1, -1) 4\n#eval overLightSpeed (0, 2) 4", "env": 7}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 6, "column": 20},
   "endPos": {"line": 6, "column": 21},
   "data":
   "Variable name `v` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _v\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "warning",
   "pos": {"line": 6, "column": 36},
   "endPos": {"line": 6, "column": 37},
   "data":
   "Variable name `p` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _p\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "false"}],
 "env": 8}

**Attendu et anti-piège.** Après correction : `true`, `false`, `false`. Piège classique : comparer `|dx| > p` au lieu de `2 * max > p` — alors `(0, 2)`/`4` rendrait `false` pour la bonne raison mais `(2, 0)`/`3` passerait aussi… par chance ; le seuil de la Loi II pour la composante orthogonale est `c/2`, pas `c`. Un second piège : oublier que `Int.natAbs` rend un `Nat` — la comparaison finale se fait entre `Nat`, sans conversion dans l'autre sens.

### Exercice 2 — Prouver une impossibilité locale par `decide`

Le blinker horizontal (`blinker_h`, prouvé oscillateur de période 2 dans `Conway.Life`) **ne peut pas** être un translateur `(1, 0)` de période 2 : un pattern qui revient à lui-même ne peut pas à la fois osciller sur place et se déplacer.

**A vous** : démontrez le fait local ci-dessous — la preuve tient en un mot.

In [10]:
-- Exercice 2 : le blinker n'est pas un translateur (1, 0)/2
-- TODO etudiant : remplacer sorry
-- Indice : l'enonce est booleen ; le moteur le reduit tout seul.
theorem blinker_not_translator :
    (!isSpaceship blinker_h 2 (1, 0)) = true := by sorry

-- Exercice 2 : le blinker n'est pas un translateur (1, 0)/2
-- TODO etudiant : remplacer sorry
-- Indice : l'enonce est booleen ; le moteur le reduit tout seul.
theorem blinker_not_translator :
        ──────────────────────▶ 🟨 declaration uses `sorry`
    (!isSpaceship blinker_h 2 (1, 0)) = true := by sorry
--% env 9
--% prove 0

Raw input:
{"cmd": "-- Exercice 2 : le blinker n'est pas un translateur (1, 0)/2\n-- TODO etudiant : remplacer sorry\n-- Indice : l'enonce est booleen ; le moteur le reduit tout seul.\ntheorem blinker_not_translator :\n    (!isSpaceship blinker_h 2 (1, 0)) = true := by sorry", "env": 8}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 5, "column": 51},
   "goal": "⊢ (!isSpaceship blinker_h 2 (1, 0)) = true",
   "endPos": {"line": 5, "column": 56}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 4, "column": 8},
   "endPos": {"line": 4, "column": 30},
   "data": "declaration uses `sorry`"}],
 "env": 9}

**Attendu et anti-piège.** La preuve attendue est `decide` — le kernel évalue `evolve 2 blinker_h` (qui redonne le blinker, période 2) et `shift (1, 0) blinker_h` (trois cellules décalées), constate l'inégalité, et réduit la négation à `true`. Anti-piège : ne pas confondre `(!isSpaceship ...) = true` (ce qui est demandé : la négation vaut vrai) avec `isSpaceship ... = false` — les deux sont interchangeables ici, mais c'est vous qui devez voir **pourquoi** (négation booléenne, pas propositionnelle).

### Exercice 3 — Étendre le certificat à un nouveau vecteur

Le module certifie `(2, 0)`/3 — vitesse `2/3 c`. Le vecteur `(0, 3)` de période 3 est le cas extrême : vitesse `c` exactement, la frontière absolue de la Loi II. Le module sait trancher ; à vous de l'invoquer.

**A vous** : complétez `noLightSpeedTranslator` pour qu'il certifie qu'aucun pattern de la famille (3, 5) ne translate de `(0, 3)` en 3 générations.

In [11]:
-- Exercice 3 : certificat pour le vecteur (0, 3)/3 (vitesse c exactement)
-- TODO etudiant : remplacer le corps ci-dessous
-- Indice : noTranslator prend (b, k), le vecteur, puis la periode.
def noLightSpeedTranslator : Bool :=
  false

-- Attendu : true
#eval noLightSpeedTranslator

-- Exercice 3 : certificat pour le vecteur (0, 3)/3 (vitesse c exactement)
-- TODO etudiant : remplacer le corps ci-dessous
-- Indice : noTranslator prend (b, k), le vecteur, puis la periode.
def noLightSpeedTranslator : Bool :=
  false

-- Attendu : true
#eval noLightSpeedTranslator
─────▶  false
--% env 10

Raw input:
{"cmd": "-- Exercice 3 : certificat pour le vecteur (0, 3)/3 (vitesse c exactement)\n-- TODO etudiant : remplacer le corps ci-dessous\n-- Indice : noTranslator prend (b, k), le vecteur, puis la periode.\ndef noLightSpeedTranslator : Bool :=\n  false\n\n-- Attendu : true\n#eval noLightSpeedTranslator", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "false"}],
 "env": 10}

**Attendu et anti-piège.** Après correction : `true`. Attention à l'ordre des arguments — `noTranslator b k v p` : famille d'abord (`3 5`), vecteur (`(0, 3)`), période (`3`). L'anti-piège de lecture : ce `true` certifie la famille (3, 5) seulement — le lemme `family_scope` (section 7) délimite exactement ce que cela couvre ; il ne dit rien d'un hypothétique pattern de taille 6 à cheval sur deux boîtes.

## 10. Conclusion — ce que le `-b` ajoute à GOL-08

GOL-08 laissait la certification formelle dehors ; ce carnet l'a faite entrer en réduisant les trois gestes du carnet Python à des théorèmes du lake `conway_lean` :

1. **Témoin** — `translator_witness_glider` : la famille (3, 5) contient un translateur `(1, -1)`/4, et c'est le glider (légitimité : `glider_mem_family`).
2. **Impossibilité scopée** — `no_translator_two_thirds_c` : aucun des 381 patterns de (3, 5) ne translate `(2, 0)`/3 (2/3 c, au-delà de la borne orthogonale).
3. **Frontière de famille** — `no_small_orthogonal_translator` contre `lwss_spaceship` : la famille (3, 3) est vide de translateurs `(0, 2)`/4 alors que le LWSS — taille 9 — le réalise ; le certificat vaut sa portée, exactement.

La marche est la même que celle gravie depuis GOL-01 : chaque barreau de l'échelle des témoins gagne son statut définitif quand il passe du code exécuté au théorème prouvé. Le prochain barreau (GOL-09) change de question : la perplexité structurelle des trajectoires.

***

**Navigation** : [<< GOL-08 Loi II sur substrat Life](GOL-08-Loi-II-Translateur-Life.ipynb) | [Index](../README.md) | [GOL-09 Perplexite structurelle >>](GOL-09-Perplexite-Structurelle-Hashlife.ipynb)

*Module du lake : [`Conway/Life/Translateur.lean`](../../../SymbolicAI/Lean/conway_lean/Conway/Life/Translateur.lean) (canonique FR) et [`Conway/Life/Translateur_en.lean`](../../../SymbolicAI/Lean/conway_lean/Conway/Life/Translateur_en.lean) (miroir EN, convention i18n EPIC #4980).*